# Top-k teacher logits on T4 (plain transformers)

Same artefact as the vLLM and TPU attempts, with none of their failure modes.
vLLM 0.34 aborts in its own warmup on a compute-7.5 card (`minimax_m3` imports
`cutlass`); the TPU path failed six times on `torch_xla`/`libtpu`. A teacher dump
is **one forward pass per window** — a serving engine is not needed.

Turing has no bf16, so fp16. Alignment is explicit rather than inferred:
`logits[i-1]` predicts token `i`, so the record for position `i` carries that
distribution and `tools/verify_topk.py` can check it directly.


In [ ]:
import os, subprocess
smi = subprocess.run(['bash','-lc','nvidia-smi -L'], capture_output=True, text=True).stdout
print(smi)
assert 'T4' in smi, 'no T4 in this session'

SMOKE = os.environ.get('SMOKE', '1') == '1'
TEACHER = 'Qwen/Qwen3-4B' if SMOKE else 'Qwen/Qwen3-8B'
K = 8 if SMOKE else 32
WINDOW = 1024
# The smoke exists to produce a verifiable artefact quickly, not a big one.
MAX_TOKENS = 200_000 if SMOKE else 22_700_000
print('teacher', TEACHER, 'k', K, 'budget', MAX_TOKENS)


In [ ]:
import glob, struct
from pathlib import Path

# SimpleTokenizer layout, verified against src/model/tokenizer.rs: ids 0-2 are
# <unk>/<eos>/<pad> and byte b lives at id b+3. Decoding therefore maps id-3, not
# id: `id & 0xFF` shifts every byte by three and breaks UTF-8 at the first
# non-ASCII character.
SPECIALS = {0: '<unk>', 1: '<eos>', 2: '<pad>'}

def find(p):
    return sorted(glob.glob(f'/kaggle/input/**/{p}', recursive=True))

TRAIN = find('train-*.bin') or find('*.bin')
assert TRAIN, 'attach sectionme/ferrisres-corpus-release-slice'
print('train shards:', len(TRAIN))

def read_u16(path):
    raw = Path(path).read_bytes()
    assert len(raw) % 2 == 0, f'{path}: odd byte count'
    return list(struct.unpack('<%dH' % (len(raw) // 2), raw))

def byte_decode(ids):
    out = bytearray()
    specials = 0
    for i in ids:
        if 3 <= i <= 258:
            out.append(i - 3)
        elif i == 1:                     # <eos> at a window boundary
            out.extend(b'\n')
            specials += 1
        elif i == 2:                     # <pad>
            specials += 1
        else:
            out.extend(b'?')
            specials += 1
    # The builder packs windows by token and concatenates documents, so a
    # multi-byte character can be split at a window boundary. `replace` keeps the
    # text decodable and the alignment check valid; it costs a few U+FFFD.
    return bytes(out).decode('utf-8', errors='replace'), specials

probe, sp = byte_decode(read_u16(TRAIN[0])[:50_000])
print('probe chars:', len(probe), 'special ids:', sp)
print('probe head:', repr(probe[:160]))
assert len(probe) > 1000, 'decode produced too little text'


In [ ]:
import json, time
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.backends.cuda.matmul.allow_tf32 = True
tok = AutoTokenizer.from_pretrained(TEACHER)
model = AutoModelForCausalLM.from_pretrained(
    TEACHER, torch_dtype=torch.float16, device_map={'': 0},
    attn_implementation='sdpa',
).eval()
print('loaded', TEACHER, 'on', next(model.parameters()).device)

out_dir = Path('/kaggle/working/topk'); out_dir.mkdir(exist_ok=True)
for old in out_dir.glob('part-*.bin'):
    old.unlink()

total = 0; part = 0; parts = []; t0 = time.perf_counter()
for shard in TRAIN:
    if total >= MAX_TOKENS:
        break
    text, specials = byte_decode(read_u16(shard))
    print('special ids in shard:', specials)
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc['input_ids'], enc['offset_mapping']
    print(shard, 'teacher tokens', len(ids))
    for i in range(0, len(ids), WINDOW):
        if total >= MAX_TOKENS:
            break
        chunk = ids[i:i + WINDOW]
        if len(chunk) < 2:
            break
        with torch.inference_mode():
            out = model(input_ids=torch.tensor([chunk], device='cuda'))
        logits = out.logits[0].float()          # [n, vocab]
        vals, idx = logits.topk(K, dim=-1)
        vals, idx = vals[:-1], idx[:-1]         # logits[i-1] predicts token i
        n = vals.shape[0]
        rec = np.zeros(n, dtype=[('idx', '<i4', (K,)), ('val', '<f4', (K,)), ('off', '<i4')])
        rec['idx'] = idx.cpu().numpy().astype('<i4')
        rec['val'] = vals.cpu().numpy().astype('<f4')
        rec['off'] = np.array([offs[i + 1 + j][0] for j in range(n)], dtype='<i4')
        assert (np.diff(rec['off']) >= 0).all(), 'offsets must be non-decreasing'
        name = f'part-{part:05d}.bin'
        rec.tofile(out_dir / name)
        parts.append({'part': name, 'tokens': int(n), 'shard': Path(shard).name})
        total += n; part += 1
        if part % 20 == 0:
            print(total, 'tokens', round(time.perf_counter() - t0, 0), 's')
        del logits

elapsed = time.perf_counter() - t0
manifest = {
    'teacher': TEACHER, 'k': K, 'dtype': 'float16', 'window': WINDOW,
    'record': 'idx:int32,val:logit:float32,off:int32; logits[i-1] predicts token i',
    'tokens': total, 'parts': parts,
    'bytes': sum((out_dir / p['part']).stat().st_size for p in parts),
    'run_seconds': round(elapsed, 2),
    'tokens_per_s': round(total / elapsed, 1) if elapsed else 0.0,
    'smoke': SMOKE,
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps({k: v for k, v in manifest.items() if k != 'parts'}, indent=2))
